In [11]:
import numpy as np
import pandas as pd
import re
import sqlite3
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from collections import Counter


In [12]:
class DataPreprocessor:
    def __init__(self, db_path):
        self.db_path = db_path

    def load_and_preprocess_data(self, max_samples):
        df = self.merge_to_dataframe()
        df = self.filter_and_sample(df, max_samples)
        df = self.compute_scores(df)
        df = self.remove_outliers(df)
        df, scaler = self.scale_scores(df)
        return df, scaler

    def merge_to_dataframe(self):
        print("🔗 Merging tweets with user metadata into DataFrame...")
        with sqlite3.connect(self.db_path) as conn:
            df = pd.read_sql_query('''
                SELECT t.clean_text AS tweet, t.likes, u.followers
                FROM tweets t
                LEFT JOIN users u ON t.user = u.user
                WHERE t.clean_text IS NOT NULL AND t.likes IS NOT NULL AND u.followers IS NOT NULL
            ''', conn)
        print(f"🧠 Merged DataFrame created with {len(df):,} rows.")
        return df

    # Additional methods for filtering, sampling, computing scores, removing outliers, and scaling

In [13]:
class ModelTrainer:
    def __init__(self, model, device):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-4)

    def train_model(self, train_loader, X_test_tensor, y_test_tensor, num_epochs=50):
        best_val_loss = float('inf')
        patience = 3
        wait = 0
        best_model_state = None

        for epoch in range(num_epochs):
            self.model.train()
            running_loss = 0.0
            for batch_x, batch_y in train_loader:
                self.optimizer.zero_grad()
                outputs = self.model(batch_x)
                loss = self.criterion(outputs, batch_y)
                loss.backward()
                self.optimizer.step()
                running_loss += loss.item()
            train_loss = running_loss / len(train_loader)

            val_loss = self.evaluate(X_test_tensor, y_test_tensor)
            print(f"Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                wait = 0
                best_model_state = self.model.state_dict()
            else:
                wait += 1
                if wait >= patience:
                    print("⏹️ Early stopping triggered!")
                    break

        if best_model_state:
            self.model.load_state_dict(best_model_state)
        return self.model

    def evaluate(self, X_test_tensor, y_test_tensor):
        self.model.eval()
        with torch.no_grad():
            val_preds = self.model(X_test_tensor)
            val_loss = self.criterion(val_preds, y_test_tensor).item()
        return val_loss

In [14]:
def save_model(model, path):
    torch.save(model.state_dict(), path)

def load_model(model, path):
    model.load_state_dict(torch.load(path))
    return model

In [15]:
if __name__ == "__main__":
    db_path = "../../data/tweets.db"
    max_samples = 280000
    device = "cuda" if torch.cuda.is_available() else "cpu"

    data_preprocessor = DataPreprocessor(db_path)
    df, scaler = data_preprocessor.load_and_preprocess_data(max_samples)

    # Embedding and dataset preparation would go here

    model = PopularityRegressor()
    trainer = ModelTrainer(model, device)
    trained_model = trainer.train_model(train_loader, X_test_tensor, y_test_tensor)

    # Evaluation and saving model

🔗 Merging tweets with user metadata into DataFrame...
🧠 Merged DataFrame created with 283,543 rows.


AttributeError: 'DataPreprocessor' object has no attribute 'filter_and_sample'